# Lab 7 - Human-in-the-Loop Checkpoint

**Section 9 - Permission Policies & Safety**  ·  **Estimated time:** 20-25 min  ·  **Estimated cost:** a few cents

This is a **Jupyter Notebook**. Run it top to bottom in **Udemy Labs** (nothing to install) or on your own machine. Read the note above each cell, then run the cell with Shift+Enter.

> **Heads up:** this lab is interactive. A few cells call `input()` to ask you to approve or deny a gated action. When a cell shows a text box, type your answer and press Enter.

## Goal
Build an agent that pauses for explicit human approval before every `bash` command and before any `write` to a path outside `/tmp/`. You set a toolset default of `always_allow` so read-only tools run freely, then override `bash` and `write` to `always_ask`. In your client you handle the server-side confirmation flow: the session pauses in a `requires_action` state, you prompt the operator, and you reply with a `user.tool_confirmation` that either allows the action or denies it with a message the model can learn from.

By the end you will have seen all three behaviors live: an approval prompt before a gated action, an `allow` that resumes the run, and a `deny` whose message reroutes the agent toward a safer approach.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("ANTHROPIC_API_KEY configured for this notebook session.")

In [ ]:
import os
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
TOOLSET = "agent_toolset_20260401"
client = Anthropic()
print("SDK ready, model:", MODEL)

### Step 1 - Create the agent with a selective `always_ask` policy
Set the toolset `default_config` to `always_allow`, then use `configs` to override `bash` and `write` to `always_ask`. This is the common production shape: let safe tools run, gate the mutating ones.

In [ ]:
agent = client.beta.agents.create(
    name="HITL Coding Assistant",
    model=MODEL,
    system=(
        "You are a careful coding assistant. Explain what you will do "
        "before doing it, then take one action at a time. Never assume "
        "approval: wait for the confirmation result. If an action is "
        "denied, read the reason and propose a safer alternative."
    ),
    tools=[{
        "type": TOOLSET,
        "default_config": {"permission_policy": {"type": "always_allow"}},
        "configs": [
            {"name": "bash", "permission_policy": {"type": "always_ask"}},
            {"name": "write", "permission_policy": {"type": "always_ask"}},
        ],
    }],
    betas=BETAS,
)
print("agent.id =", agent.id)

### Step 2 - Write the policy: auto-allow `/tmp/`, ask otherwise
The policy decides what to do with each pending tool_use. Writes under `/tmp/` are low-risk scratch space, so let them through. Everything else escalates to a prompt that returns `allow`, a bare `deny`, or a `deny` with a reason. The reason becomes the `deny_message` the model sees on its next turn.

In [ ]:
def decide(event_id, recent_events):
    """Return ('allow' | 'deny', optional_deny_message) for a pending tool_use."""
    # Find the agent.tool_use event the session is blocking on.
    target = next(
        (e for e in recent_events if getattr(e, "id", None) == event_id),
        None,
    )
    if target is None:
        # Defensive: if we cannot identify the tool, refuse rather than approve.
        return "deny", "Internal error: missing tool_use event."

    # Auto-allow writes under /tmp/ (low-risk scratch space).
    if target.name == "write":
        path = (target.input or {}).get("path", "")
        if path.startswith("/tmp/"):
            print(f"\n[auto-allow] write to {path} (under /tmp/)")
            return "allow", None

    # Escalate to the human for bash and for writes outside /tmp/.
    print(f"\n[approval needed] {target.name} -> {target.input}")
    ans = input("  approve? [y / N / explain <reason>] ").strip().lower()
    if ans == "y":
        return "allow", None
    if ans.startswith("explain"):
        reason = ans[len("explain"):].strip()
        return "deny", reason or "I'd rather not, please try another way."
    # Any other input (N, blank, garbage) is a hard deny.
    return "deny", "Denied by the operator. Do not retry this exact action."

### Step 3 - Create the environment and session
Create a plain unrestricted cloud environment and one session bound to the agent.

In [ ]:
env = client.beta.environments.create(
    name="hitl-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=env.id,
    title="HITL demo",
    betas=BETAS,
)
print("session.id =", session.id)

### Step 4 - Drive the session and handle `requires_action`
Stream the session. When you reach `session.status_idle` with a `requires_action` stop reason, run `decide()` for each blocking event id and send one `user.tool_confirmation` per id. Send a confirmation for **every** id, or the session deadlocks. We keep a running `recent` list so `decide()` can look up the tool_use behind each id.

The default task writes to `/workspace/hello.py` (outside `/tmp/`) and then runs it, so both gated tools fire. You will be prompted twice. Try approving both with `y` the first time.

In [ ]:
def drive(prompt):
    recent = []
    with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
        client.beta.sessions.events.send(
            session.id,
            events=[{
                "type": "user.message",
                "content": [{"type": "text", "text": prompt}],
            }],
            betas=BETAS,
        )
        for event in stream:
            recent.append(event)

            if event.type == "agent.message":
                for b in event.content:
                    if b.type == "text":
                        print(b.text, end="", flush=True)

            elif event.type == "agent.tool_use":
                print(f"\n[tool requested: {event.name}]")

            elif event.type == "session.status_idle":
                sr = getattr(event, "stop_reason", None)
                if sr and sr.type == "requires_action":
                    # Answer every blocking id, or the session deadlocks.
                    for eid in sr.event_ids:
                        choice, msg = decide(eid, recent)
                        body = {
                            "type": "user.tool_confirmation",
                            "tool_use_id": eid,
                            "result": choice,
                        }
                        if choice == "deny" and msg:
                            body["deny_message"] = msg
                        client.beta.sessions.events.send(
                            session.id, events=[body], betas=BETAS,
                        )
                    # Loop continues: the session resumes and streams more.
                else:
                    # Idle with no pending action means the turn is done.
                    break
    print("\n\nDone.")

drive(
    "Write a hello-world Python script to /workspace/hello.py "
    "and then run it with python."
)

### Step 5 - Exercise the deny path
Run the same task again, but this time **deny** the `bash` step. Type `explain not on prod` at the prompt (or just `N`) and watch the next `agent.message` acknowledge the denial and propose a safer approach instead of retrying the same command.

In [ ]:
drive(
    "Write a hello-world Python script to /workspace/hello.py "
    "and then run it with python."
)

### Step 6 - See the auto-allow rule (optional)
Point the task at `/tmp/` instead. The write passes with an `[auto-allow]` line and no prompt; only the `bash` run pauses for confirmation.

In [ ]:
drive(
    "Write a hello-world Python script to /tmp/hello.py "
    "and then run it with python."
)

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
- An `agent.id` and a `session.id` line near the top.
- A `[tool requested: write]` line, then an approval prompt for the write to `/workspace/hello.py` (because it is outside `/tmp/`).
- A `[tool requested: bash]` line, then an approval prompt before the run.
- If you `allow` both: the script is written, runs, and prints hello-world; the session goes idle and the cell prints `Done.`
- If you `deny` the `bash` step with a message: the next `agent.message` acknowledges the denial and proposes another approach instead of retrying the same command.
- For the `/tmp/` task: an `[auto-allow]` line appears for the write with no prompt; only `bash` pauses.

## Troubleshooting
- **No prompt appears.** Check the policy config shape in Step 1. The override lives in `configs` as `{"name": "bash", "permission_policy": {"type": "always_ask"}}`, and the `default_config` must be `always_allow`. If the task never needs `bash` or `write`, the model will not invoke them. Use a task that clearly requires both.
- **Session hangs after a confirmation.** You must send a `user.tool_confirmation` for *every* id in `requires_action.event_ids`. Miss one and the session stays paused. Loop over all the ids before continuing.
- **`requires_action` never fires.** Make sure you read `stop_reason` off the `session.status_idle` event (`getattr(event, "stop_reason", None)`) and check `sr.type == "requires_action"`. An idle event with no such stop reason means the turn finished; that is your signal to `break`.
- **`deny_message` looks ignored.** It is delivered to the model as context for the next turn, not echoed back to you. Read the following `agent.message` to see the adaptation. Only attach `deny_message` on the `deny` path.
- **The `input()` box does not appear.** In some Jupyter front ends the prompt renders at the top of the notebook or in the kernel status bar. Click into the box, type, and press Enter. Re-run the cell if a stream was interrupted.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste:

> "Build me a Managed Agents agent called 'HITL Coding Assistant' on `claude-haiku-4-5-20251001` (`betas=['managed-agents-2026-04-01']`, all calls on `client.beta.*`). Use the `agent_toolset_20260401` toolset with a `default_config` of `always_allow`, but override `bash` and `write` to `always_ask` via `configs`. Give it a system prompt telling it to explain before acting, take one action at a time, never assume approval, and propose a safer alternative when an action is denied."

Then:

> "Create an unrestricted cloud environment and start a session. Stream it, and whenever the session hits `session.status_idle` with a `requires_action` stop reason, for each blocking event id either auto-allow `write` when the path starts with `/tmp/`, or prompt me at stdin. Send one `user.tool_confirmation` per id, using `deny_message` when I deny. Ask it to write `/workspace/hello.py` and run it."

Then drive it interactively:

> "Run it. Approve the write but deny the bash step with the message 'not on prod', and show me how the agent reroutes."

Compare what Claude Code writes to the cells above.

## Stretch
- **Path-aware write rule.** Already wired in: auto-allow `write` when the path starts with `/tmp/`, otherwise ask. Extend it to a small allowlist of safe prefixes (for example `/tmp/` and `/workspace/scratch/`).
- **Denylist risky bash.** Before prompting, auto-deny any `bash` input matching `rm -rf`, `curl ... | sh`, or writes to system paths, with a clear `deny_message`.
- **Route approvals to Slack.** Replace the stdin prompt with a Slack message and button, and log every allow/deny decision to a CSV for audit.